# scVI best - scib_sim1

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'scib_sim1'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('Batch',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'Group'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'scib_sim1'
assert BATCH_KEYS == ('Batch',)
assert LABEL_KEY == 'Group'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                                               scib_sim1
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                              [Batch]
label_key                                                 Group
metrics       [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=12097 n_batches=6 n_labels=7 counts_layer=True
ISOLISI preflight: min_batch_coverage=2 isolated_labels=['Group7']
CPCR preflight: eligible_labels=7


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
OPTUNA_BEST_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/scib_sim1/expanded_metrics_v1/scvi_scib_sim1_joint_expanded_metrics_v1_best_params.json'
OPTUNA_TRIALS_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/scib_sim1/expanded_metrics_v1/scvi_scib_sim1_joint_expanded_metrics_v1_trials.csv'
OPTUNA_STUDY_NAME = 'scvi_scib_sim1_joint_expanded_metrics_v1'
OPTUNA_BEST_TRIAL = 130
OPTUNA_BEST_VALUE = 0.8709494815189158
OPTUNA_BEST_PARAMS = {'n_latent': 61, 'n_hidden': 64, 'n_layers': 1, 'dropout_rate': 0.11944530674951662, 'lr': 0.007377571973642884}

with open(OPTUNA_BEST_SOURCE, "r", encoding="utf-8") as source_file:
    current_source = json.load(source_file)
assert current_source["study_name"] == OPTUNA_STUDY_NAME
assert current_source["dataset_name"] == DATASET_NAME
assert current_source["best_value"] == OPTUNA_BEST_VALUE
assert current_source["best_params"] == OPTUNA_BEST_PARAMS
assert tuple(OPTUNA_BEST_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

SCVI_PARAMS = dict(OPTUNA_BEST_PARAMS)

pd.Series(
    {
        "source": OPTUNA_BEST_SOURCE,
        "trials_source": OPTUNA_TRIALS_SOURCE,
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="best_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.9032455179686841


optimised clustering against scvi_metric_label
optimal cluster resolution: 0.2
optimal score: 1.0
BRAS...


seed           9.000000
BAMI           1.000000
CASW           0.638349
SCGRAPHM       0.435665
ISOLISI        1.000000
BRAS           0.961915
KBET           0.799589
CPCR           0.979126
bio_score      0.768504
batch_score    0.913543
joint_score    0.826520
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,1.0,0.632551,0.714935,1.0,0.968430,0.812810,0.984958,0.836872,0.922066,0.870949
1,1,1.0,0.628119,0.502986,1.0,0.955960,0.726427,0.975342,0.782776,0.885910,0.824030
2,2,1.0,0.647368,0.412149,1.0,0.954524,0.774846,0.978099,0.764879,0.902490,0.819923
3,3,1.0,0.633757,0.282624,1.0,0.953987,0.730892,0.975782,0.729095,0.886887,0.792212
4,4,1.0,0.658206,0.665287,1.0,0.947352,0.639553,0.966011,0.830873,0.850972,0.838913
5,5,1.0,0.664556,0.730019,1.0,0.948903,0.694733,0.967931,0.848644,0.870523,0.857395
6,6,1.0,0.654421,0.635878,1.0,0.942839,0.710236,0.970243,0.822575,0.874439,0.843321
7,7,1.0,0.653563,0.636551,1.0,0.947582,0.665460,0.970195,0.822529,0.861079,0.837949
8,8,1.0,0.645748,0.567953,1.0,0.949405,0.704242,0.964432,0.803425,0.872693,0.831132
9,9,1.0,0.638349,0.435665,1.0,0.961915,0.799589,0.979126,0.768504,0.913543,0.826520


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,1.000000,0.000000,1.000000,1.000000
CASW,0.645664,0.011538,0.628119,0.664556
SCGRAPHM,0.558405,0.138982,0.282624,0.730019
ISOLISI,1.000000,0.000000,1.000000,1.000000
BRAS,0.953090,0.007237,0.942839,0.968430
KBET,0.725879,0.053103,0.639553,0.812810
CPCR,0.973212,0.006186,0.964432,0.984958
bio_score,0.801017,0.036396,0.729095,0.848644
batch_score,0.884060,0.021663,0.850972,0.922066
joint_score,0.834234,0.020363,0.792212,0.870949
